In [1]:
#

In [2]:
import pandas as pd
import regex as re
import os
from pathlib import Path

In [3]:
current_dir = str(Path.cwd().parents[2])

In [4]:
import sys
sys.path.insert(0, current_dir)

In [5]:
from maqam_prompt_generator import generate_prompt

In [6]:
df = pd.read_json('./majnoon_layla.json', encoding='utf-8')

In [7]:
from django.utils.text import slugify
df['section_no'] = df['sections'].apply(lambda x: x['section_id']).astype(int)
df['title'] = df['sections'].apply(lambda x: x['title'])
df['title'] = df.apply(lambda x: (f"{str(x['section_no']).zfill(2)} {x['title']}"), axis=1)
df['title'] = df['title'].apply(lambda x: slugify(x, allow_unicode=True))
df['mood'] = df['sections'].apply(lambda x: x['mood'])
df

,sections,section_no,title,mood
0,"{'section_id': 1, 'maqam': 'Nahawand', 'mood':...",1,01-نار-ليلى-في-ثمدين-واسترجاع-الخوالي,"nostalgic, gentle sorrow, reflective"
1,"{'section_id': 2, 'maqam': 'Kurd', 'mood': 'so...",2,02-عهد-الصبا-واعتراض-القدر-والوشاة,"solemn grief, fate, tragic weight"
2,"{'section_id': 3, 'maqam': 'Ajam', 'mood': 'de...",3,03-سلطان-الهوى-والتحدي-بالقوافي,"defiant passion, noble, enduring"
3,"{'section_id': 4, 'maqam': 'Hijaz', 'mood': 'd...",4,04-ذروة-الوجد-ومناجاة-حمامة-العقيق,"deep yearning, poignant ache, mystical"
4,"{'section_id': 5, 'maqam': 'Kurd', 'mood': 'st...",5,05-داء-الهيام-ونحول-الجسد-ورثاء-الصحب,"stately sorrow, somber, haunting"
5,"{'section_id': 6, 'maqam': 'Nahawand', 'mood':...",6,06-التيه-في-الفضاء-وركب-اليمانيين,"desert wandering, wistful, tender"
6,"{'section_id': 7, 'maqam': 'Hijaz', 'mood': 't...",7,07-طلب-الأكفان-والوداع-التراجيدي-الأخير,"tragic elegy, ultimate farewell, grief"


In [8]:
df['lyrics'] = df['sections'].apply(lambda x: x['lyrics'])

In [9]:
df['maqam'] = df['sections'].apply(lambda x: x['maqam'])

In [10]:
from string import ascii_lowercase

In [11]:
ascii_lowercase

'abcdefghijklmnopqrstuvwxyz'

In [12]:
df['start_on'] = df['lyrics'].apply(lambda x: [i for i in str(x).splitlines() if len(set(i).intersection(ascii_lowercase))==0][1])

In [13]:
df['start_on'] = df['start_on'].apply(lambda x: ' '.join(x.split()[:3]).strip())

In [14]:
df['PROMPT'] = df.apply(lambda x: generate_prompt(choice=x['maqam'], start_phrase=x['start_on'], mood=x['mood']), axis=1)

In [15]:
df['style'] = df['PROMPT'].apply(lambda x: x.split('PROMPT:\n\n```\n')[1].split("\n```\n\nEXCLUDE:\n\n```\n")[0])
df['exclude'] = df['PROMPT'].apply(lambda x: x.split('PROMPT:\n\n```\n')[1].split("\n```\n\nEXCLUDE:\n\n```\n")[1].removesuffix('```'))

In [16]:
df['workspace_url'] = input("Enter Suno Workspace URL:\n")
df['vocal_gender'] = 'male'


In [17]:
df.sample()

,sections,section_no,title,mood,lyrics,maqam,start_on,PROMPT,style,exclude,workspace_url,vocal_gender
0,"{'section_id': 1, 'maqam': 'Nahawand', 'mood':...",1,01-نار-ليلى-في-ثمدين-واسترجاع-الخوالي,"nostalgic, gentle sorrow, reflective",///***///\n[Intro | single clean guitar | clos...,Nahawand,تَذَكَّرْتُ لَيْلَى وَالسِّنِينَ,PROMPT:\n\n```\n[Is_MAX_MODE: MAX](MAX) [QUALI...,[Is_MAX_MODE: MAX](MAX) [QUALITY: MAX](MAX) [R...,"Oud, Qanun, Darbuka, Tabla, Ney, Buzuq, Sitar,...",https://suno.com/create?wid=6baf0657-4ec6-44b3...,male


In [18]:
df['style'] = df['PROMPT'].apply(lambda x: x.split('PROMPT:\n\n```\n')[1].split("\n```\n\nEXCLUDE:\n\n```\n")[0])
df['exclude'] = df['PROMPT'].apply(lambda x: x.split('PROMPT:\n\n```\n')[1].split("\n```\n\nEXCLUDE:\n\n```\n")[1].removesuffix('```'))

In [19]:
df.to_pickle('./majnoon_layla.pkl', protocol=4)

In [20]:
idx = 0

In [21]:
print(f'''
{df['lyrics'][idx]}
---
{df['style'][idx]}
---
{df['exclude'][idx]}

''')


///***///
[Intro | single clean guitar | close-mic'd, plate reverb, short decay]
تَذَكَّرْتُ لَيْلَى وَالسِّنِينَ الْخَوَالِيَا
وَأَيَّامَ لَا نَخْشَى عَلَى ٱلَّهۡوِ نَاهِيَا...

[Verse 1 | wistful resonant vocals | muted rhythmic guitars and sparse bass]
تَذَكَّرْتُ لَيْلَى وَالسِّنِينَ الْخَوَالِيَا
وَأَيَّامَ لَا نَخْشَى عَلَى ٱلَّهۡوِ نَاهِيَا...
وَيَوْمٍ كَظِلِّ الرُّمْحِ قَصَّرْتُ ظِلَّهُ
بِلَيْلَى فَلَهَّانِي وَمَا كُنْتُ لَاهِيَا...
بِثَمْدَيْنِ لَاحَتْ نَارُ لَيْلَى وَصُحْبَتِي
بِذَاتِ الْغَضَى تُزْجِي الْمَطِيَّ النَّوَاجِيَا...
فَقَالَ بَصِيرُ الْقَوْمِ أَلْمَحْتُ كَوْكَبًا
بَدَا فِي سَوَادِ ٱلَّيۡلِ فَرْدًا يَمَانِيَا...
فَقُلْتُ لَهُ بَلْ نَارُ لَيْلَى تَوَقَّدَتْ
بِعَلْيَا تَسَامَى ضَوْؤُهَا فَبَدَا لِيَا...

[Chorus | emotional soaring vocals | full orchestral swell and guitar build — Nahawand]
فَمَا أُشْرِفُ الْأَيْفَاعَ إِلَّا صَبَابَةً
وَلَا أُنْشِدُ الْأَشْعَارَ إِلَّا تَدَاوِيَا...
وَقَدْ يَجْمَعُ اللَّهُ الشَّتِيتَيْنِ بَعْدَمَا
يَظُنَّانِ كُلَّ الظَّنِّ أَنْ لَا 